In [ ]:
# ==========================================================================
# 금융 콜센터 고객 문의 유형 분류 웹 화면 (Gradio) — 셀 하나로 실행
# --------------------------------------------------------------------------
# [전체 흐름]
#   0. 도구 설치          : kiwipiepy, koreanize-matplotlib, gradio
#   1. 전처리 함수        : 대화 파일 → [A번호] 단위로 대화 나누기 → 중복 제거
#                           → 인사를 건너뛴 '고객 첫 질문' 뽑기 → Kiwi로 단어 나누기
#   2. 학습 함수 (train)  : 정답 파일과 합치기 → 학습 80% / 평가 20% 나누기
#                           → TF-IDF → 로지스틱 회귀(균형) 학습 → 성적표·혼동 행렬
#   3. 응답 함수 (ask)    : 새 문의 → 같은 방법으로 단어 나누기 → 유형별 확률(응답)
#                           → 과거 문의 중 가장 비슷한 5개 찾기(검색 결과)
#   4. 웹 화면 실행       : ① 파일 선택 → ② 학습 결과 → ③ 문의 분류·검색 순서로 배치
#
# [사용 순서]
#   ① 이 셀 실행 → 셀 아래(또는 https://….gradio.live 주소)에 웹 화면이 나타남
#   ② 상담 대화 .txt 파일과 정답 파일 inquiry_labels.csv 를 고르고 → [학습하기] (30초 정도)
#   ③ 고객 문의 문장을 입력하고 → [분류하기] (Enter 가능)
#   끝낼 때: 셀 왼쪽의 ■(중지) 버튼
#
# [주의] 공개 주소(….gradio.live)는 주소를 아는 누구나 접속할 수 있음
# ==========================================================================


# ----- 0. 도구 설치 -----
# PIL(Pillow): Colab에 이미 설치된 이미지 도구, 지금 버전 번호를 알아내려고 불러옴
import PIL

# pip 설치 (pillow=={PIL.__version__}: 설치 중에 Pillow 버전이 바뀌어 ImportError가 나는 것을 막으려고 지금 버전으로 고정)
!pip install -q kiwipiepy koreanize-matplotlib gradio pillow=={PIL.__version__}


# ----- 도구 불러오기 -----
import re
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib                       # 불러오기만 하면 그래프 한글 글꼴(나눔고딕)이 설정됨
from kiwipiepy import Kiwi
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

# 한국어 형태소 분석기
kiwi = Kiwi()

# 유형 순서 (표와 그래프에서 늘어놓을 순서)
ORDER = ["은행·카드", "자동차보험", "질병·상해 보험금", "재물·배상 보험", "보험 계약·보험료"]


# ----- 1. 전처리 함수 -----
# ✏️ 인사 발화에서 지울 표현 (지우고 나서 남는 글자가 없으면 '인사뿐인 발화')
#    |는 '또는', [\s.?!,~]는 공백 문자나 . ? ! , ~ 중 하나
GREETING = r"안녕하세요|안녕하십니까|수고하십니다|수고많으십니다|여보세요|반갑습니다|보험|은행|카드|콜센터|고객센터|이지요|이죠|인가요|맞나요|네|[\s.?!,~]"

# ✏️ 남길 품사: NNG 일반명사, NNP 고유명사, VV 동사, VA 형용사, XR 어근, MAG 부사, SL 외국어
KEEP = {"NNG", "NNP", "VV", "VA", "XR", "MAG", "SL"}


# 대화 하나에서 고객 첫 질문을 찾는 함수
def first_question(dialog):
    for line in dialog.splitlines():
        if not line.startswith("고객:"):
            continue                                       # 상담사 발화는 건너뛰기
        q = line[3:]                                       # '고객:' 세 글자 떼기
        q = re.sub(r"O{2,}|0{3,}", "", q)                  # 가림 표시(OO, 000) 지우기
        q = re.sub(r"\s+", " ", q).strip()                 # 띄어쓰기 정리
        if len(q) >= 5 and re.sub(GREETING, "", q) != "":  # 5글자 이상이고 인사뿐인 발화가 아니면
            return q                                       # 그 발화를 첫 질문으로 돌려줌
    return ""                                              # 질문이 없으면 빈 글자


# 문장 하나 → 띄어쓰기로 이은 단어 목록 (t.tag.split("-")[0]: 'VA-I' → 'VA')
def to_words(sentence):
    return " ".join(t.form for t in kiwi.tokenize(sentence) if t.tag.split("-")[0] in KEEP)


# 대화 파일 글자 전체 → [ID, 첫질문, 단어] 표
def make_table(text):
    # '[A숫자]' 줄을 기준으로 자르기 → ['', 'A4075', '대화 내용', 'A4076', '대화 내용', ...]
    parts = re.split(r"^\[(A\d+)\]\s*$", text, flags=re.M)
    # parts[1::2] = 번호들, parts[2::2] = 대화 내용들 → 두 열짜리 표
    df = pd.DataFrame({"ID": parts[1::2], "원문": [p.strip() for p in parts[2::2]]})
    # 완전히 같은 대화 지우기
    df = df.drop_duplicates("원문").reset_index(drop=True)
    # 대화마다 고객 첫 질문, 그 첫 질문의 단어 목록
    df["첫질문"] = df["원문"].apply(first_question)
    df["단어"] = df["첫질문"].apply(to_words)
    return df


# ----- 2. 학습 함수: [학습하기] 버튼 -----
def train(txt_file, label_file):
    # 파일 두 개를 다 고르지 않았으면 안내 메시지만 돌려줌 (나머지 출력은 비움)
    if txt_file is None or label_file is None:
        return "상담 대화 .txt 파일과 정답 파일(inquiry_labels.csv)을 모두 선택하세요.", None, None, None

    # 대화 파일 읽기 → 전처리 표
    df = make_table(open(txt_file, encoding="utf-8").read())

    # 정답 파일(열: ID, 유형)과 합치기: 두 표에 모두 있는 ID만 남김, 첫 질문이 같은 문의는 하나만
    data = df.merge(pd.read_csv(label_file), on="ID", how="inner")
    data = data.drop_duplicates("첫질문").reset_index(drop=True)
    if len(data) == 0:
        return "대화 번호(ID)와 정답 파일이 맞지 않습니다. 파일을 확인하세요.", None, None, None

    # 80%는 학습용, 20%는 평가용 (stratify: 유형 비율을 똑같이, random_state: 매번 같게)
    X_train, X_test, y_train, y_test = train_test_split(
        data["단어"], data["유형"], test_size=0.2, random_state=42, stratify=data["유형"])

    # TF-IDF: 학습용으로 단어 사전을 만들고(fit) 숫자로 바꾸기, 평가용은 바꾸기만
    # (analyzer=str.split: 이미 띄어쓰기로 나눠 둔 단어를 그대로 사용)
    tfidf = TfidfVectorizer(analyzer=str.split)
    A_train = tfidf.fit_transform(X_train)
    A_test = tfidf.transform(X_test)

    # 로지스틱 회귀 학습 (class_weight="balanced": 개수가 적은 유형을 틀리면 더 크게 벌점)
    model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(A_train, y_train)

    # 평가용으로 예측하고 정확도·macro-F1 계산
    pred = model.predict(A_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="macro")

    # 유형별 성적표 (output_dict=True: 사전으로 받아 표로 바꿈) → ORDER 유형만, 소수 셋째 자리
    report = pd.DataFrame(classification_report(y_test, pred, labels=ORDER, output_dict=True)).T
    report = report.loc[ORDER].round(3).reset_index().rename(columns={"index": "유형", "support": "평가 문의 수"})
    report["평가 문의 수"] = report["평가 문의 수"].astype(int)            # 120.0 → 120 (소수점 없애기)

    # 혼동 행렬 그림 (가로 = 예측, 세로 = 정답, 대각선 = 맞힌 문의 수)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=ORDER, cmap="Blues",
                                            colorbar=False, ax=ax, xticks_rotation=30)
    ax.set_xlabel("예측")
    ax.set_ylabel("정답")
    ax.set_title("혼동 행렬 (평가용)")
    plt.tight_layout()
    plt.close(fig)                                         # 셀 아래에 따로 그려지지 않게 닫기 (웹 화면에만 표시)

    # 처리 결과 메시지
    message = (f"문의 {len(data):,}건으로 학습 완료 (학습 {len(X_train):,} / 평가 {len(X_test):,}). "
               f"평가용 정확도 {acc:.1%}, macro-F1 {f1:.3f}")

    # 검색용: 전체 문의를 TF-IDF 숫자로 바꿔 둠
    A_all = tfidf.transform(data["단어"])

    # 메시지, 성적표, 혼동 행렬, 보관함(응답 함수가 쓸 도구와 데이터 묶음)을 돌려줌
    return message, report, fig, (tfidf, model, data, A_all)


# ----- 3. 응답 함수: [분류하기] 버튼 -----
def ask(question, store):
    # 학습 전이거나 문의가 비어 있으면 안내 메시지
    if store is None:
        return "먼저 파일 두 개를 고르고 [학습하기]를 누르세요.", None, None
    if not question.strip():
        return "고객 문의 문장을 입력하세요.", None, None

    # 보관함에서 TF-IDF 도구, 모델, 데이터, 전체 문의 숫자 행렬 꺼내기
    tfidf, model, data, A_all = store

    # 새 문의도 학습 때와 같은 방법으로 단어 나누기 → TF-IDF 숫자로 바꾸기
    words = to_words(question)
    A_new = tfidf.transform([words])

    # 유형별 확률 (predict_proba: 0~1 → %), 모델의 유형 순서를 ORDER 순서로 다시 정렬
    proba = pd.Series(model.predict_proba(A_new)[0] * 100, index=model.classes_)[ORDER]
    best = proba.idxmax()                                  # 확률이 가장 큰 유형

    # 응답 글
    answer = (f"예측 유형: {best} ({proba[best]:.1f}%)\n"
              f"분석에 쓴 단어: {words if words else '(없음 — 분석할 단어가 없어 예측이 불확실함)'}")

    # 유형별 확률 표
    proba_table = proba.round(1).reset_index()
    proba_table.columns = ["유형", "확률(%)"]

    # 검색 결과: 새 문의와 TF-IDF 코사인 유사도가 가장 높은 과거 문의 5개
    scores = cosine_similarity(A_new, A_all)[0]
    top = scores.argsort()[::-1][:5]                       # 유사도 큰 순서로 5개 위치
    top = top[scores[top] > 0]                             # 겹치는 단어가 하나도 없는(유사도 0) 문의는 빼기
    similar = data.loc[top, ["ID", "첫질문", "유형"]].assign(유사도=scores[top].round(3))

    return answer, proba_table, similar


# ----- 4. 웹 화면 -----
# gr.Blocks(): 웹 화면 전체 틀, with 블록 안의 부품이 위에서부터 차례로 배치됨
with gr.Blocks() as demo:
    gr.Markdown("# 📞 금융 콜센터 문의 유형 분류")
    gr.Markdown("고객의 **첫 질문**으로 문의 유형(은행·카드 / 자동차보험 / 질병·상해 보험금 / 재물·배상 보험 / 보험 계약·보험료)을 "
                "자동으로 분류하고, 비슷한 과거 문의를 찾아 줍니다. (Kiwi + TF-IDF + 로지스틱 회귀)")

    # 보이지 않는 보관함: train이 만든 (TF-IDF, 모델, 데이터, 숫자 행렬)을 담아 ask에 넘겨줌
    store = gr.State()

    # ===== ① 파일 선택 =====
    gr.Markdown("## ① 파일 선택")
    with gr.Row():                                         # gr.Row(): 안의 부품을 가로로 나란히
        txt_file = gr.File(label="상담 대화 파일 (.txt)", file_types=[".txt"])
        label_file = gr.File(label="정답 파일 (inquiry_labels.csv)", file_types=[".csv"])
    train_btn = gr.Button("학습하기", variant="primary")
    status = gr.Textbox(label="처리 결과")

    # ===== ② 학습 결과 =====
    gr.Markdown("## ② 학습 결과 (평가용 20%)")
    report = gr.Dataframe(label="유형별 성적표 (precision 정밀도 · recall 재현율 · f1-score)")
    cm_plot = gr.Plot(label="혼동 행렬")

    # ===== ③ 문의 분류와 비슷한 문의 검색 =====
    gr.Markdown("## ③ 문의 분류하기")
    question = gr.Textbox(label="고객 문의", placeholder="예) 카드를 잃어버렸는데 정지해 주세요")
    # ✏️ 예시 문의: 누르면 입력 칸이 채워짐
    gr.Examples(examples=["카드를 잃어버렸는데 정지해 주세요", "주차하다가 옆 차를 긁었어요",
                          "도수치료 받았는데 실비 청구 되나요?", "아랫집 천장에 물이 샌다고 연락이 왔어요",
                          "보험료가 두 달 밀렸는데 어떻게 되나요?"], inputs=question)
    ask_btn = gr.Button("분류하기", variant="primary")
    answer = gr.Textbox(label="응답", lines=2)
    with gr.Row():
        proba_table = gr.Dataframe(label="유형별 확률", scale=1)
        similar = gr.Dataframe(label="검색 결과: 비슷한 과거 문의 5개", wrap=True, scale=3,
                               column_widths=["12%", "56%", "20%", "12%"])   # 열 너비 (ID, 첫질문, 유형, 유사도)

    # 버튼과 함수 연결: 함수가 돌려준 값이 outputs 부품에 순서대로 표시됨
    train_btn.click(train, inputs=[txt_file, label_file], outputs=[status, report, cm_plot, store])
    ask_btn.click(ask, inputs=[question, store], outputs=[answer, proba_table, similar])
    question.submit(ask, inputs=[question, store], outputs=[answer, proba_table, similar])   # Enter로도 실행

# 웹 화면 실행 (debug=True: 셀이 계속 실행되며 오류를 셀 아래에 보여 줌)
demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 10.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 45.5 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a16fae7096974e6f9c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (htt